In [2]:
pip install sacrebleu evaluate sacremoses

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 19.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.5.1
    Uninstalling fsspec-2025.5.1:
      Successfully uninstalled fsspec-2025.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system ==

In [3]:
import os
import torch
from datasets import load_dataset
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, EarlyStoppingCallback
from datetime import datetime
import evaluate
import numpy as np

In [3]:
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_DATASETS_CACHE"] = "./hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "./transformers_cache"
os.environ["WANDB_DISABLED"] = "true"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU only")

raw_datasets = load_dataset("wmt14", "fr-en")

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

def has_both_languages(example):
    t = example.get("translation")
    return t is not None and t.get("en") is not None and t.get("fr") is not None

clean_train_dataset = raw_datasets["train"].filter(has_both_languages).shuffle(seed=42)
small_train_dataset = clean_train_dataset.select(range(100000))
small_val_dataset = clean_train_dataset.select(range(100000, 125000))

def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True, padding="max_length")
    return model_inputs

train_dataset = small_train_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
val_dataset = small_val_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")

bleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu_score = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    meteor_score = meteor.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": bleu_score["score"], "meteor": meteor_score["meteor"]}

CUDA available: True
Using GPU: Tesla P100-PCIE-16GB


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

train-00000-of-00030.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

train-00001-of-00030.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

...

train-00021-of-00030.parquet:   0%|          | 0.00/264M [00:00<?, ?B/s]

train-00022-of-00030.parquet:   0%|          | 0.00/267M [00:00<?, ?B/s]

train-00023-of-00030.parquet:   0%|          | 0.00/270M [00:00<?, ?B/s]

train-00024-of-00030.parquet:   0%|          | 0.00/274M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [18]:
training_args = Seq2SeqTrainingArguments(
    run_name=f"marianmt-wmt14-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}",
    output_dir="./outputs_marianmt_wmt14",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    num_train_epochs=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=50,
    save_total_limit=1,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipykernel_36/4257672725.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
print("Starting training...")
trainer.train()
trainer.save_model("./translation1")
tokenizer.save_pretrained("./translation1")
!zip -r translation1.zip ./translation1

Starting training...


Epoch,Training Loss,Validation Loss


In [4]:
import os
import torch
from datasets import load_dataset
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    PrinterCallback,
)
from datetime import datetime
import evaluate
import numpy as np
import time


In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    PrinterCallback,
)
from datetime import datetime
import evaluate
import numpy as np
import time

# Silence warnings and set local cache paths
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"
os.environ["HF_DATASETS_CACHE"] = "./hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "./transformers_cache"

# Show device info
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU only")

# Load dataset
print("Loading dataset...")
raw_datasets = load_dataset("wmt14", "fr-en")

# Filter out bad samples
def has_both_languages(example):
    t = example.get("translation")
    return t is not None and t.get("en") is not None and t.get("fr") is not None

clean_train_dataset = raw_datasets["train"].filter(has_both_languages).shuffle(seed=42)

# Use smaller subset
print("Selecting 10K samples for train, 2.5K for val")
small_train_dataset = clean_train_dataset.select(range(50000))
small_val_dataset = clean_train_dataset.select(range(50000, 60000))

# Load tokenizer and model
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

# Preprocessing
def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    return tokenizer(inputs, text_target=targets, max_length=128, truncation=True, padding="max_length")

print("Tokenizing datasets...")
start = time.time()
train_dataset = small_train_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
val_dataset = small_val_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
print("Tokenization completed in", round(time.time() - start, 2), "seconds.")

# Set PyTorch format
train_dataset.set_format("torch")
val_dataset.set_format("torch")

# Load evaluation metrics
bleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu_score = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    meteor_score = meteor.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": bleu_score["score"], "meteor": meteor_score["meteor"]}

# Training arguments
training_args = Seq2SeqTrainingArguments(
    run_name=f"marianmt-wmt14-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}",
    output_dir="./outputs_marianmt_wmt14",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    num_train_epochs=15,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=1,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    disable_tqdm=False,
)

# Trainer setup
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[PrinterCallback(), EarlyStoppingCallback(early_stopping_patience=1)],
)

# Train with error handling
try:
    print("Training started...")
    trainer.train()
    print("Training complete.")
except Exception as e:
    print("Training failed:", e)

# Save model
trainer.save_model("./translation1")
tokenizer.save_pretrained("./translation1")

# Zip the folder for download
!zip -r translation1.zip ./translation1


In [5]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
# Load dataset
print("Loading dataset...")
raw_datasets = load_dataset("wmt14", "fr-en")

# Filter out bad samples
def has_both_languages(example):
    t = example.get("translation")
    return t is not None and t.get("en") is not None and t.get("fr") is not None

clean_train_dataset = raw_datasets["train"].filter(has_both_languages).shuffle(seed=42)

# Use smaller subset
print("Selecting 10K samples for train, 2.5K for val")
small_train_dataset = clean_train_dataset.select(range(50000))
small_val_dataset = clean_train_dataset.select(range(50000, 60000))
def preprocess_function(examples):
    inputs = ["translate English to French: " + ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    return tokenizer(inputs, text_target=targets, max_length=128, truncation=True, padding="max_length")

model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

print("Tokenizing datasets...")
start = time.time()
train_dataset = small_train_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
val_dataset = small_val_dataset.map(preprocess_function, batched=True, remove_columns=["translation"])
print("Tokenization completed in", round(time.time() - start, 2), "seconds.")

# Set PyTorch format
train_dataset.set_format("torch")
val_dataset.set_format("torch")

# Load evaluation metrics
bleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    bleu_score = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    meteor_score = meteor.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": bleu_score["score"], "meteor": meteor_score["meteor"]}
# Training arguments
training_args = Seq2SeqTrainingArguments(
    run_name=f"marianmt-wmt14-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}",
    output_dir="./outputs_marianmt_wmt14",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),
    num_train_epochs=15,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=1,
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    disable_tqdm=False,
)

# Trainer setup
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[PrinterCallback(), EarlyStoppingCallback(early_stopping_patience=1)],
)

# Train with error handling
try:
    print("Training started...")
    trainer.train()
    print("Training complete.")
except Exception as e:
    print("Training failed:", e)

# Save model
trainer.save_model("./translation1")
tokenizer.save_pretrained("./translation1")

# Zip the folder for download
!zip -r translation1.zip ./translation1


Loading dataset...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

train-00000-of-00030.parquet:   0%|          | 0.00/252M [00:00<?, ?B/s]

train-00001-of-00030.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

...

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Bleu,Meteor
1,0.454700,0.391671,7.343758,0.302399
2,0.453700,0.389444,7.411311,0.303635
3,0.431400,0.388693,7.405455,0.303532
4,0.470000,0.387745,7.426476,0.303941
5,0.497600,0.388033,7.423904,0.303792


{'loss': 10.6487, 'grad_norm': 32.327476501464844, 'learning_rate': 1.9992322456813822e-05, 'epoch': 0.0064}
{'loss': 9.5422, 'grad_norm': 30.58890151977539, 'learning_rate': 1.998379185327362e-05, 'epoch': 0.0128}
{'loss': 8.2756, 'grad_norm': 37.54740524291992, 'learning_rate': 1.997526124973342e-05, 'epoch': 0.0192}
{'loss': 7.6107, 'grad_norm': 41.42388916015625, 'learning_rate': 1.996673064619322e-05, 'epoch': 0.0256}
{'loss': 6.3249, 'grad_norm': 43.31626892089844, 'learning_rate': 1.995820004265302e-05, 'epoch': 0.032}
{'loss': 5.3355, 'grad_norm': 40.30341339111328, 'learning_rate': 1.994966943911282e-05, 'epoch': 0.0384}
{'loss': 4.1105, 'grad_norm': 40.976444244384766, 'learning_rate': 1.9941138835572617e-05, 'epoch': 0.0448}
{'loss': 3.4497, 'grad_norm': 35.919586181640625, 'learning_rate': 1.993260823203242e-05, 'epoch': 0.0512}
{'loss': 2.622, 'grad_norm': 29.53961944580078, 'learning_rate': 1.9924077628492218e-05, 'epoch': 0.0576}
{'loss': 1.9615, 'grad_norm': 13.74830436

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


{'train_runtime': 5357.2643, 'train_samples_per_second': 139.997, 'train_steps_per_second': 4.376, 'train_loss': 0.5308142288632677, 'epoch': 5.0}
Training complete.
  adding: translation1/ (stored 0%)
  adding: translation1/model.safetensors (deflated 9%)
  adding: translation1/tokenizer_config.json (deflated 94%)
  adding: translation1/generation_config.json (deflated 29%)
  adding: translation1/added_tokens.json (deflated 83%)
  adding: translation1/spiece.model (deflated 48%)
  adding: translation1/config.json (deflated 62%)
  adding: translation1/training_args.bin (deflated 52%)
  adding: translation1/special_tokens_map.json (deflated 85%)


In [6]:
trainer.save_model("./translation1")
tokenizer.save_pretrained("./translation1")
!zip -r translation1.zip ./translation1

updating: translation1/ (stored 0%)
updating: translation1/model.safetensors (deflated 9%)
updating: translation1/tokenizer_config.json (deflated 94%)
updating: translation1/generation_config.json (deflated 29%)
updating: translation1/added_tokens.json (deflated 83%)
updating: translation1/spiece.model (deflated 48%)
updating: translation1/config.json (deflated 62%)
updating: translation1/training_args.bin (deflated 52%)
updating: translation1/special_tokens_map.json (deflated 85%)


In [7]:
import shutil

# Move zip file to /kaggle/working (if not already there)
shutil.move("./translation1.zip", "/kaggle/working/translation1.zip")
print("Download from the sidebar: translation1.zip")


Download from the sidebar: translation1.zip
